In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

In [0]:
matches_df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("/Volumes/ipl_analytics/bronze/raw_data/matches.csv")
)
matches_df.display()

In [0]:
display(matches_df)

#Add Metadata Columns

In [0]:
matches_bronze_df = (
    matches_df
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_system", lit("CSV"))
    .withColumn("source_file", lit("matches.csv"))
)

In [0]:
display(matches_bronze_df)

In [0]:
%sql
DROP TABLE IF EXISTS ipl_analytics.bronze.matches_raw;

In [0]:
(matches_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ipl_analytics.bronze.matches_raw"))

In [0]:
display(spark.table("ipl_analytics.bronze.matches_raw"))

#sql verfication


In [0]:
%sql

SELECT *
FROM ipl_analytics.bronze.matches_raw
LIMIT 10;

In [0]:
%sql
DROP TABLE IF EXISTS ipl_analytics.bronze.teams_raw;


In [0]:
from pyspark.sql.types import *

team_schema = StructType([
    StructField("team_id", StringType(), True),
    StructField("team_name", StringType(), True),
    StructField("coach", StringType(), True),
    StructField("home_ground", StringType(), True),
    StructField("captain", StringType(), True)
])

teams_df = (
    spark.read
    .option("header", "true")
    .schema(team_schema)
    .csv("/Volumes/ipl_analytics/bronze/raw_data/teams.csv")
)
teams_df = (
    teams_df
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_system", lit("CSV"))
    .withColumn("source_file", lit("teams.csv"))
)
teams_df.printSchema()
display(teams_df)

(teams_df.write
        .format("delta") 
        .mode("overwrite").saveAsTable("ipl_analytics.bronze.teams_raw"))


# -----------------------------------------
# Reusable Bronze Ingestion Function
# -----------------------------------------

In [0]:
%sql
DROP TABLE IF EXISTS ipl_analytics.bronze.matches_raw;
DROP TABLE IF EXISTS ipl_analytics.bronze.ball_by_ball_raw;
DROP TABLE IF EXISTS ipl_analytics.bronze.players_raw;

In [0]:
def bronze_ingestion(file_name, table_name):

    file_path = f"/Volumes/ipl_analytics/bronze/raw_data/{file_name}"

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(file_path)
    )

    bronze_df = (
        df.withColumn("ingestion_ts", current_timestamp())
          .withColumn("source_system", lit("CSV"))
          .withColumn("source_file", lit(file_name))
    )

    (bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"ipl_analytics.bronze.{table_name}"))

    print(f"✅ {table_name} created successfully")

In [0]:
bronze_ingestion("matches.csv", "matches_raw")
bronze_ingestion("players.csv", "players_raw")
bronze_ingestion("ball_by_ball.csv", "ball_by_ball_raw")

In [0]:
%sql
SHOW TABLES IN ipl_analytics.bronze;

In [0]:
spark.table("ipl_analytics.bronze.players_raw").display()

In [0]:
%sql
SELECT 'matches_raw' AS table_name, COUNT(*) AS row_count
FROM ipl_analytics.bronze.matches_raw

UNION ALL

SELECT 'players_raw', COUNT(*)
FROM ipl_analytics.bronze.players_raw

UNION ALL

SELECT 'teams_raw', COUNT(*)
FROM ipl_analytics.bronze.teams_raw

UNION ALL

SELECT 'ball_by_ball_raw', COUNT(*)
FROM ipl_analytics.bronze.ball_by_ball_raw;